In [0]:
%sql
-- Create a separate schema for SDP (Declarative Pipelines)
CREATE SCHEMA IF NOT EXISTS capstone_supply_chain.sdp_demo

In [0]:
#sdp validation
from pyspark.sql import functions as F

C = "capstone_supply_chain"

sdp = spark.table(f"{C}.sdp_demo.gold_vendor_spend_by_currency")
nb = spark.table(f"{C}.gold.agg_vendor_spend_by_currency")

# 1. Row counts of every pipeline table
for t in ["bronze_purchase_orders", "silver_purchase_orders", "gold_vendor_spend_by_currency"]:
    print(f"sdp_demo.{t}: {spark.table(f'{C}.sdp_demo.{t}').count()} rows")

print("notebook gold table (agg_vendor_spend_by_currency):", nb.count(), "rows")

# 2. Compare the two results in both directions (same columns, same order)
cols = ["vendor_id", "currency", "po_count", "total_spend"]
a = sdp.select(*cols)
b = nb.select("vendor_id", "currency", "po_count", F.round("total_spend", 2).alias("total_spend"))

only_in_pipeline = a.exceptAll(b).count()
only_in_notebook = b.exceptAll(a).count()

print("rows only in the pipeline result:", only_in_pipeline)
print("rows only in the notebook result:", only_in_notebook)

if only_in_pipeline == 0 and only_in_notebook == 0:
    print("✅ The pipeline and the notebook produce identical results")
else:
    print("❌ There are differences, showing them below")
    a.exceptAll(b).show()
    b.exceptAll(a).show()